# Set Signatures as a Dot Plot

A [`SetCollection`](../../python/set/api.md) derives feature spaces from the
cells in each set. `calc_signature` calculates each set's mean expression and,
optionally, the fraction of its cells that express each gene, in one modality.
A Clustergram shows both as a dot plot: color for expression, size for the
fraction expressing. This brief example uses **simulated** cells of six types.

In [1]:
import anndata as ad
import numpy as np
import pandas as pd

import celldega as dega

## Simulate cells

In [2]:
rng = np.random.default_rng(0)

cell_types = ["B cells", "CD4 T", "CD8 T", "Myeloid", "Epithelial", "Stromal"]
markers = {
    "B cells": ["MS4A1", "CD79A"],
    "CD4 T": ["CD3E", "CD4"],
    "CD8 T": ["CD3E", "CD8A"],
    "Myeloid": ["CD68", "LYZ"],
    "Epithelial": ["EPCAM", "KRT8"],
    "Stromal": ["COL1A1", "DCN"],
}
genes = sorted({gene for genes_ in markers.values() for gene in genes_})

n_cells = 3000
cell_type = rng.choice(cell_types, n_cells, p=[0.12, 0.15, 0.13, 0.15, 0.30, 0.15])
counts = rng.poisson(0.3, (n_cells, len(genes))).astype(float)
for i, kind in enumerate(cell_type):
    for gene in markers[kind]:
        counts[i, genes.index(gene)] = rng.poisson(6)

adata = ad.AnnData(
    counts,
    obs=pd.DataFrame(
        {"cell_type": pd.Categorical(cell_type, categories=cell_types)},
        index=[f"cell_{i}" for i in range(n_cells)],
    ),
    var=pd.DataFrame(index=genes),
)
adata

AnnData object with n_obs × n_vars = 3000 × 11
    obs: 'cell_type'

## Calculate the signatures

One row per cell type. `fraction_expressing_layer` stores the fraction of each
set's cells with counts above zero beside the expression, and
`rank_genes_groups=True` ranks each set's marker genes with Scanpy.

In [3]:
sets = dega.SetCollection(adata, set_col="cell_type")
sets.calc_signature(
    adata,
    modality_name="expression",
    fraction_expressing_layer="fraction_expressing",
    rank_genes_groups=True,
    verbose=False,
)
expression = sets.mod["expression"]
expression.to_df().round(1)

,CD3E,CD4,CD68,CD79A,CD8A,COL1A1,DCN,EPCAM,KRT8,LYZ,MS4A1
cell_type,,,,,,,,,,,
Epithelial,9.9,9.9,10.0,9.9,10.0,10.0,9.9,12.9,12.9,9.9,9.9
CD4 T,12.9,12.9,9.9,9.9,9.9,10.0,10.0,10.0,10.0,10.0,9.9
B cells,9.8,9.9,9.8,12.9,10.0,9.8,9.7,10.0,10.0,10.0,12.9
Stromal,10.0,9.9,9.9,10.0,9.9,12.9,12.9,10.0,10.0,9.9,9.8
Myeloid,9.9,9.9,12.9,9.9,9.9,9.9,9.9,10.0,10.0,12.9,9.9
CD8 T,12.9,9.9,10.0,9.8,12.9,10.0,10.0,10.0,9.9,9.9,9.9


## Show them as a dot plot

`Matrix(mdata=..., color_by=..., size_by_layer=...)` takes the expression for
color and the fraction expressing for dot size from the same modality.

In [4]:
mat = dega.clust.Matrix(mdata=sets, color_by="expression", size_by_layer="fraction_expressing")
mat.cluster()
cgm = dega.viz.Clustergram(matrix=mat, width=350, height=400, row_label_scale=0.6, col_label_scale=0.45)

In [5]:
cgm

The marker ranks travel with the collection, in
`sets.mod["expression"].uns["rank_genes_groups"]`, and save with it:

```python
sets.write("cell_types.h5mu")
dega.SetCollection.read("cell_types.h5mu")
```